In [4]:
# ============================================================
# D12 — Validation C
# 1. Imports
# ============================================================

from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import hashlib
import json
import platform
import re
import sys
import unicodedata

import pandas as pd

In [5]:
# ============================================================
# 2. Configuration
# ============================================================

DOCUMENT_ID = "D12"
DOCUMENT_NAME = "Our World in Data — Annual CO2 emissions time series"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
INPUT_REPRESENTATION = (
    "Complete deterministically normalised structural Markdown table"
)

EXPECTED_SOURCE_SHA256 = (
    "eb46e8e036c08032bbd18d234d3da22442111288e26af32d3eacef592457e788"
)

EXPECTED_RECORD_COUNT = 25

REFERENCE_CATEGORY = "Environmental time-series"
REFERENCE_TOPIC = "Annual CO2 emissions"
REFERENCE_DESCRIPTION = "Annual CO2 emissions"
REFERENCE_UNIT = None

TARGET_YEARS = [
    1750, 1800, 1850, 1900, 1950,
    1960, 1970, 1980, 1990, 2000,
    2010, 2011, 2012, 2013, 2014,
    2015, 2016, 2017, 2018, 2019,
    2020, 2021, 2022, 2023, 2024
]

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

# Frozen from final D12 Validation A and reused unchanged.
IDENTITY_FIELDS = [
    "Reporting Period"
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Source Location"
]

EXPECTED_CATEGORY_COUNTS = {
    REFERENCE_CATEGORY: EXPECTED_RECORD_COUNT
}

OUTPUT_DIR = Path("outputs_D12_validation_C_revised")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "detailed":
        OUTPUT_DIR / "D12_branch_C_validation_detailed.csv",

    "discrepant":
        OUTPUT_DIR / "D12_branch_C_discrepant_records.csv",

    "missing":
        OUTPUT_DIR / "D12_branch_C_missing_records.csv",

    "unsupported":
        OUTPUT_DIR / "D12_branch_C_unsupported_records.csv",

    "alignment_issues":
        OUTPUT_DIR / "D12_branch_C_alignment_issues.json",

    "reference_semantics":
        OUTPUT_DIR / "D12_reference_semantics_confirmation.json",

    "summary":
        OUTPUT_DIR / "D12_branch_C_validation_summary.json",

    "conclusion":
        OUTPUT_DIR / "D12_branch_C_validation_conclusion.json"
}

print("Output directory:", OUTPUT_DIR)

Output directory: outputs_D12_validation_C_revised


In [7]:
# ============================================================
# 3. Upload canonical Validation C inputs
# ============================================================
# Required:
#   1) D12_reference_values.csv
#   2) D12_branch_C_structure_check.json
#   3) D12_branch_C_experiment_metadata.json
#   4) D12_branch_C_normalisation_check.json
#   5) D12_branch_C_experiment_summary.json
#
# Required only when the preserved Branch C response is evaluable:
#   6) D12_branch_C_parsed_extraction.json
#
# The experiment summary is checked first. If the execution was not
# content-evaluable, record- and field-level metrics must not be forced.

print(
    "Upload:\n"
    "1. D12_reference_values.csv\n"
    "2. D12_branch_C_structure_check.json\n"
    "3. D12_branch_C_experiment_metadata.json\n"
    "4. D12_branch_C_normalisation_check.json\n"
    "5. D12_branch_C_experiment_summary.json\n"
    "6. D12_branch_C_parsed_extraction.json if it was created"
)

uploaded = files.upload()
uploaded_paths = [Path(name) for name in uploaded]

csv_paths = [
    path for path in uploaded_paths
    if path.suffix.lower() == ".csv"
]

json_paths = [
    path for path in uploaded_paths
    if path.suffix.lower() == ".json"
]

if len(csv_paths) != 1:
    raise ValueError(
        "Upload exactly one CSV file: D12_reference_values.csv."
    )

REFERENCE_PATH = csv_paths[0]

EXTRACTION_PATH = None
STRUCTURE_PATH = None
METADATA_PATH = None
NORMALISATION_PATH = None
EXPERIMENT_SUMMARY_PATH = None


def canonical_filename(path):
    return path.name.casefold().replace(" ", "_")


# First pass: canonical filename patterns.
for path in json_paths:
    filename = canonical_filename(path)

    if "d12_branch_c_parsed_extraction" in filename:
        EXTRACTION_PATH = path
        continue

    if "d12_branch_c_structure_check" in filename:
        STRUCTURE_PATH = path
        continue

    if (
        "d12_branch_c_experiment_metadata" in filename
        and "_pre" not in filename
    ):
        METADATA_PATH = path
        continue

    if (
        "d12_branch_c_normalisation_check" in filename
        or "d12_branch_c_normalization_check" in filename
    ):
        NORMALISATION_PATH = path
        continue

    if "d12_branch_c_experiment_summary" in filename:
        EXPERIMENT_SUMMARY_PATH = path
        continue


# Second pass: content-based fallback.
for path in json_paths:
    with path.open("r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        EXPERIMENT_SUMMARY_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "parsed_extraction_created" in obj
        and "records_evaluable" in obj
        and "validation_status" in obj
    ):
        EXPERIMENT_SUMMARY_PATH = path
        continue

    if (
        NORMALISATION_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and obj.get("parent_branch") == "B"
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_PATH = path
        continue

    if (
        METADATA_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "source_sha256" in obj
        and "raw_response_sha256" in obj
        and "structure_check_file" in obj
    ):
        METADATA_PATH = path
        continue

    if (
        STRUCTURE_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structure_valid" in obj
        and "records_evaluable" in obj
        and "validation_status" not in obj
    ):
        STRUCTURE_PATH = path
        continue

    if (
        EXTRACTION_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        EXTRACTION_PATH = path
        continue


for label, path in {
    "structure check": STRUCTURE_PATH,
    "experiment metadata": METADATA_PATH,
    "normalisation check": NORMALISATION_PATH,
    "experiment summary": EXPERIMENT_SUMMARY_PATH,
}.items():

    if path is None:
        raise ValueError(
            f"Could not identify required {label} file."
        )


with EXPERIMENT_SUMMARY_PATH.open(
    "r",
    encoding="utf-8"
) as f:
    experiment_summary_payload = json.load(f)


content_evaluable = bool(
    experiment_summary_payload.get("records_evaluable")
    and experiment_summary_payload.get("parsed_extraction_created")
)


if content_evaluable and EXTRACTION_PATH is None:
    raise ValueError(
        "This D12 Branch C execution is content-evaluable, so "
        "D12_branch_C_parsed_extraction.json is required."
    )

if not content_evaluable:
    raise ValueError(
        "This D12 Branch C execution is not content-evaluable. "
        "Do not calculate record- or field-level metrics. "
        "Use a non-evaluable validation pathway like D9."
    )


print("\nCanonical Branch C validation inputs resolved:")
print("Reference:", REFERENCE_PATH.name)
print("Parsed extraction:", EXTRACTION_PATH.name)
print("Structure check:", STRUCTURE_PATH.name)
print("Experiment metadata:", METADATA_PATH.name)
print("Normalisation check:", NORMALISATION_PATH.name)
print("Experiment summary:", EXPERIMENT_SUMMARY_PATH.name)
print("Content evaluable:", content_evaluable)

Upload:
1. D12_reference_values.csv
2. D12_branch_C_structure_check.json
3. D12_branch_C_experiment_metadata.json
4. D12_branch_C_normalisation_check.json
5. D12_branch_C_experiment_summary.json
6. D12_branch_C_parsed_extraction.json if it was created


Saving D12_branch_C_structure_check.json to D12_branch_C_structure_check.json
Saving D12_branch_C_parsed_extraction.json to D12_branch_C_parsed_extraction.json
Saving D12_branch_C_normalisation_check.json to D12_branch_C_normalisation_check.json
Saving D12_branch_C_experiment_summary.json to D12_branch_C_experiment_summary.json
Saving D12_branch_C_experiment_metadata.json to D12_branch_C_experiment_metadata.json
Saving D12_reference_values.csv to D12_reference_values.csv

Canonical Branch C validation inputs resolved:
Reference: D12_reference_values.csv
Parsed extraction: D12_branch_C_parsed_extraction.json
Structure check: D12_branch_C_structure_check.json
Experiment metadata: D12_branch_C_experiment_metadata.json
Normalisation check: D12_branch_C_normalisation_check.json
Experiment summary: D12_branch_C_experiment_summary.json
Content evaluable: True


In [9]:
# ============================================================
# 4. Load and fingerprint inputs
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
STRUCTURE_SHA256 = sha256_file(STRUCTURE_PATH)
METADATA_SHA256 = sha256_file(METADATA_PATH)
NORMALISATION_SHA256 = sha256_file(NORMALISATION_PATH)
EXPERIMENT_SUMMARY_SHA256 = sha256_file(
    EXPERIMENT_SUMMARY_PATH
)


reference_df = pd.read_csv(
    REFERENCE_PATH,
    encoding="utf-8-sig",
    keep_default_na=False
)

with EXTRACTION_PATH.open("r", encoding="utf-8") as f:
    extraction_payload = json.load(f)

with STRUCTURE_PATH.open("r", encoding="utf-8") as f:
    structure_payload = json.load(f)

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata_payload = json.load(f)

with NORMALISATION_PATH.open("r", encoding="utf-8") as f:
    normalisation_payload = json.load(f)


extracted_records = extraction_payload.get("records", [])
extracted_df = pd.DataFrame(extracted_records)


print("Reference SHA-256:", REFERENCE_SHA256)
print("Parsed extraction SHA-256:", EXTRACTION_SHA256)
print("Structure SHA-256:", STRUCTURE_SHA256)
print("Metadata SHA-256:", METADATA_SHA256)
print("Normalisation SHA-256:", NORMALISATION_SHA256)
print("Experiment summary SHA-256:", EXPERIMENT_SUMMARY_SHA256)
print("Reference records:", len(reference_df))
print("Extracted records:", len(extracted_records))

Reference SHA-256: b3500919a265a9669b6b8af65476adca041c6e11a0b803bbc8ea28e957216b8e
Parsed extraction SHA-256: b86c256922860db81d17db3179ab02cc04366f1bf320acf90bd99375763eae58
Structure SHA-256: 82c6c545096bbd97ebfff2a57d28281c1175f0fa480f602216bbcea05fb9cc9a
Metadata SHA-256: 9f07e0aaa89f4a976c478fced2d7e11fd0923e3dca888dfe3e7da2ea832cd8a9
Normalisation SHA-256: 76381b35016f4e61af8e352fdfc69a5bb59039f2c2e39bce5e8714ed5d39a643
Experiment summary SHA-256: 99bfe78229eacb3931af332e810c14cf4041cfeeade98a181ecf88444197aadd
Reference records: 25
Extracted records: 25


In [10]:
# ============================================================
# 5. Confirm fixed Stage 1 reference semantics
# ============================================================

reference_schema_exact = (
    reference_df.columns.tolist() == EXPECTED_FIELDS
)

reference_record_count_valid = (
    len(reference_df) == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"].value_counts().to_dict()
    if "Category" in reference_df.columns
    else {}
)

reference_category_counts_valid = (
    reference_category_counts == EXPECTED_CATEGORY_COUNTS
)

reference_topic_constant = bool(
    (reference_df["Topic"] == REFERENCE_TOPIC).all()
)

reference_description_constant = bool(
    (reference_df["Description"] == REFERENCE_DESCRIPTION).all()
)

# CSV loading with keep_default_na=False turns null cells into "".
reference_unit_null = bool(
    reference_df["Unit"].apply(
        lambda value: value is None
        or (isinstance(value, str) and value.strip() == "")
        or pd.isna(value)
    ).all()
)

reference_values_numeric = pd.to_numeric(
    reference_df["Value"],
    errors="coerce"
).notna().all()

reference_values_non_negative = bool(
    (
        pd.to_numeric(reference_df["Value"], errors="coerce")
        >= 0
    ).all()
)

reference_periods = pd.to_numeric(
    reference_df["Reporting Period"],
    errors="coerce"
)

reference_years_valid = (
    reference_periods.notna().all()
    and reference_periods.astype(int).tolist() == TARGET_YEARS
)

reference_identity_unique = (
    reference_df["Reporting Period"].astype(str).nunique()
    == EXPECTED_RECORD_COUNT
)

reference_source_locations_valid = bool(
    reference_df["Source Location"]
    .astype(str)
    .str.fullmatch(r"CSV data row \d+")
    .all()
)

reference_semantic_checks = {
    "reference_schema_exact":
        bool(reference_schema_exact),
    "reference_record_count_valid":
        bool(reference_record_count_valid),
    "reference_category_counts_valid":
        bool(reference_category_counts_valid),
    "reference_topic_constant":
        bool(reference_topic_constant),
    "reference_description_constant":
        bool(reference_description_constant),
    "reference_unit_null":
        bool(reference_unit_null),
    "reference_values_numeric":
        bool(reference_values_numeric),
    "reference_values_non_negative":
        bool(reference_values_non_negative),
    "reference_years_valid":
        bool(reference_years_valid),
    "reference_identity_unique":
        bool(reference_identity_unique),
    "reference_source_locations_valid":
        bool(reference_source_locations_valid)
}

reference_semantics_valid = all(
    reference_semantic_checks.values()
)

REFERENCE_SEMANTICS_CONFIRMATION = {
    "document_id": DOCUMENT_ID,
    "reference_semantics_valid":
        bool(reference_semantics_valid),
    "checks":
        reference_semantic_checks
}

PATHS["reference_semantics"].write_text(
    json.dumps(
        REFERENCE_SEMANTICS_CONFIRMATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    REFERENCE_SEMANTICS_CONFIRMATION,
    ensure_ascii=False,
    indent=2
))

if not reference_semantics_valid:
    raise AssertionError(
        "D12 Stage 1 reference semantics are not valid."
    )

{
  "document_id": "D12",
  "reference_semantics_valid": true,
  "checks": {
    "reference_schema_exact": true,
    "reference_record_count_valid": true,
    "reference_category_counts_valid": true,
    "reference_topic_constant": true,
    "reference_description_constant": true,
    "reference_unit_null": true,
    "reference_values_numeric": true,
    "reference_values_non_negative": true,
    "reference_years_valid": true,
    "reference_identity_unique": true,
    "reference_source_locations_valid": true
  }
}


In [11]:
# ============================================================
# 6. Confirm Branch C provenance, normalisation integrity and schema
# ============================================================

top_level_object_valid = isinstance(
    extraction_payload,
    dict
)

document_id_correct = (
    extraction_payload.get("document_id")
    == DOCUMENT_ID
)

branch_correct = (
    extraction_payload.get("branch")
    == BRANCH
)

records_is_list = isinstance(
    extraction_payload.get("records"),
    list
)

record_schema_valid = True
field_types_valid = True
schema_issue_rows = []

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]


if records_is_list:

    for i, record in enumerate(extracted_records):

        if not isinstance(record, dict):
            record_schema_valid = False

            schema_issue_rows.append({
                "record_index": i,
                "issue": "record_not_object"
            })

            continue


        if list(record.keys()) != EXPECTED_FIELDS:
            record_schema_valid = False

            schema_issue_rows.append({
                "record_index": i,
                "issue": "field_names_or_order",
                "observed_fields": list(record.keys())
            })


        for field in STRING_OR_NULL_FIELDS:
            value = record.get(field)

            if (
                value is not None
                and not isinstance(value, str)
            ):
                field_types_valid = False

                schema_issue_rows.append({
                    "record_index": i,
                    "issue": "field_type",
                    "field": field,
                    "observed_type":
                        type(value).__name__
                })


        value = record.get("Value")

        if (
            value is not None
            and (
                isinstance(value, bool)
                or not isinstance(
                    value,
                    (int, float)
                )
            )
        ):
            field_types_valid = False

            schema_issue_rows.append({
                "record_index": i,
                "issue": "field_type",
                "field": "Value",
                "observed_type":
                    type(value).__name__
            })

else:
    record_schema_valid = False
    field_types_valid = False


branch_C_structure_valid = bool(
    structure_payload.get(
        "structure_valid"
    )
)

schema_validity = all([
    top_level_object_valid,
    document_id_correct,
    branch_correct,
    records_is_list,
    record_schema_valid,
    field_types_valid,
    branch_C_structure_valid
])


parsed_extraction_hash_matches_metadata = (
    metadata_payload.get(
        "parsed_extraction_sha256"
    )
    == EXTRACTION_SHA256
)

metadata_source_hash_matches_stage_1 = (
    metadata_payload.get(
        "source_sha256"
    )
    == EXPECTED_SOURCE_SHA256
)

normalisation_source_hash_matches_stage_1 = (
    normalisation_payload.get(
        "source_sha256",
        EXPECTED_SOURCE_SHA256
    )
    == EXPECTED_SOURCE_SHA256
)

parent_equivalence_passed = bool(
    normalisation_payload.get(
        "parent_equivalence_passed",
        False
    )
)

normalisation_integrity_passed = bool(
    normalisation_payload.get(
        "normalisation_integrity_passed",
        False
    )
)


representation_integrity = {
    "parent_branch":
        normalisation_payload.get(
            "parent_branch"
        ),

    "parent_equivalence_passed":
        parent_equivalence_passed,

    "normalisation_integrity_passed":
        normalisation_integrity_passed,

    "parent_row_count":
        normalisation_payload.get(
            "parent_row_count"
        ),

    "branch_C_row_count":
        normalisation_payload.get(
            "branch_C_row_count"
        ),

    "row_count_preserved":
        normalisation_payload.get(
            "row_count_preserved"
        ),

    "row_identity_and_order_preserved":
        normalisation_payload.get(
            "row_identity_and_order_preserved"
        ),

    "year_sequence_preserved":
        normalisation_payload.get(
            "year_sequence_preserved"
        ),

    "value_sequence_preserved":
        normalisation_payload.get(
            "value_sequence_preserved"
        ),

    "source_row_sequence_preserved":
        normalisation_payload.get(
            "source_row_sequence_preserved"
        ),

    "target_year_values_preserved":
        normalisation_payload.get(
            "target_year_values_preserved"
        ),

    "deterministic_representation_verified":
        normalisation_payload.get(
            "deterministic_representation_verified"
        ),

    "complete_275_row_representation_retained":
        normalisation_payload.get(
            "complete_275_row_representation_retained"
        ),

    "source_scope_filtering_applied":
        normalisation_payload.get(
            "source_scope_filtering_applied"
        ),

    "target_year_filtering_applied":
        normalisation_payload.get(
            "target_year_filtering_applied"
        ),

    "row_reordering_applied":
        normalisation_payload.get(
            "row_reordering_applied"
        ),

    "semantic_harmonisation_applied":
        normalisation_payload.get(
            "semantic_harmonisation_applied"
        ),

    "semantic_rewriting_applied":
        normalisation_payload.get(
            "semantic_rewriting_applied"
        ),

    "unit_inference_applied":
        normalisation_payload.get(
            "unit_inference_applied"
        ),

    "unit_conversion_applied":
        normalisation_payload.get(
            "unit_conversion_applied"
        ),

    "numeric_calculation_applied":
        normalisation_payload.get(
            "numeric_calculation_applied"
        ),

    "numeric_rescaling_applied":
        normalisation_payload.get(
            "numeric_rescaling_applied"
        ),

    "numeric_rounding_applied":
        normalisation_payload.get(
            "numeric_rounding_applied"
        ),

    "manual_reconstruction_applied":
        normalisation_payload.get(
            "manual_reconstruction_applied"
        ),

    "manual_correction_applied":
        normalisation_payload.get(
            "manual_correction_applied"
        ),

    "reference_values_used_for_transformation":
        normalisation_payload.get(
            "reference_values_used_for_transformation"
        )
}


input_provenance = {
    "reference_file":
        REFERENCE_PATH.name,

    "reference_sha256":
        REFERENCE_SHA256,

    "parsed_extraction_file":
        EXTRACTION_PATH.name,

    "parsed_extraction_sha256":
        EXTRACTION_SHA256,

    "structure_check_file":
        STRUCTURE_PATH.name,

    "structure_check_sha256":
        STRUCTURE_SHA256,

    "experiment_metadata_file":
        METADATA_PATH.name,

    "experiment_metadata_sha256":
        METADATA_SHA256,

    "normalisation_integrity_file":
        NORMALISATION_PATH.name,

    "normalisation_integrity_sha256":
        NORMALISATION_SHA256,

    "experiment_summary_file":
        EXPERIMENT_SUMMARY_PATH.name,

    "experiment_summary_sha256":
        EXPERIMENT_SUMMARY_SHA256,

    "branch_C_structure_valid":
        branch_C_structure_valid,

    "parsed_extraction_hash_matches_metadata":
        parsed_extraction_hash_matches_metadata,

    "source_hash_matches_stage_1":
        metadata_source_hash_matches_stage_1,

    "parent_B_equivalence_passed":
        parent_equivalence_passed,

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}


print("Schema validity:", schema_validity)
print(
    "Branch C structure valid:",
    branch_C_structure_valid
)
print(
    "Parent B equivalence passed:",
    parent_equivalence_passed
)
print(
    "Normalisation integrity passed:",
    normalisation_integrity_passed
)

print(
    json.dumps(
        input_provenance,
        ensure_ascii=False,
        indent=2
    )
)

Schema validity: True
Branch C structure valid: True
Parent B equivalence passed: True
Normalisation integrity passed: True
{
  "reference_file": "D12_reference_values.csv",
  "reference_sha256": "b3500919a265a9669b6b8af65476adca041c6e11a0b803bbc8ea28e957216b8e",
  "parsed_extraction_file": "D12_branch_C_parsed_extraction.json",
  "parsed_extraction_sha256": "b86c256922860db81d17db3179ab02cc04366f1bf320acf90bd99375763eae58",
  "structure_check_file": "D12_branch_C_structure_check.json",
  "structure_check_sha256": "82c6c545096bbd97ebfff2a57d28281c1175f0fa480f602216bbcea05fb9cc9a",
  "experiment_metadata_file": "D12_branch_C_experiment_metadata.json",
  "experiment_metadata_sha256": "9f07e0aaa89f4a976c478fced2d7e11fd0923e3dca888dfe3e7da2ea832cd8a9",
  "normalisation_integrity_file": "D12_branch_C_normalisation_check.json",
  "normalisation_integrity_sha256": "76381b35016f4e61af8e352fdfc69a5bb59039f2c2e39bce5e8714ed5d39a643",
  "experiment_summary_file": "D12_branch_C_experiment_summary.

In [12]:
# ============================================================
# 7. Comparison-only canonicalisation
# ============================================================

def normalise_text(value):
    if value is None:
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    text = (
        text.replace("\u00a0", " ")
        .replace("\u2007", " ")
        .replace("\u202f", " ")
        .replace("—", "-")
        .replace("–", "-")
        .replace("’", "'")
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def canonical_period(value):
    text = normalise_text(value)

    if text is None:
        return None

    if re.fullmatch(r"\d{4}", text):
        return int(text)

    return text


def parse_numeric(value):
    if value is None or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    text = str(value).strip().replace(",", "")

    if re.fullmatch(r"[-+]?\d+(?:\.\d+)?", text):
        return float(text)

    return None


def canonical_null(value):
    if value is None:
        return None

    if isinstance(value, float) and pd.isna(value):
        return None

    if isinstance(value, str) and value.strip() == "":
        return None

    return normalise_text(value)


def canonical_source_location(value):
    text = normalise_text(value)

    if text is None:
        return None

    match = re.fullmatch(
        r"CSV data row (\d+)",
        text
    )

    return (
        f"CSV data row {int(match.group(1))}"
        if match
        else text
    )

In [13]:
# ============================================================
# 8. Deterministic one-to-one alignment by Reporting Period
# ============================================================

def build_identity_index(records, dataset_name):
    index = {}
    duplicate_groups = []

    for record_index, record in enumerate(records):
        if not isinstance(record, dict):
            continue

        identity = canonical_period(
            record.get("Reporting Period")
        )

        if identity in index:
            duplicate_groups.append({
                "dataset": dataset_name,
                "identity": identity,
                "record_index": record_index
            })
        else:
            index[identity] = {
                "record_index": record_index,
                "record": record
            }

    return index, duplicate_groups


reference_records = reference_df.to_dict("records")

reference_index, reference_duplicates = (
    build_identity_index(
        reference_records,
        "Reference"
    )
)

extraction_index, extraction_duplicates = (
    build_identity_index(
        extracted_records,
        "Extraction"
    )
)

alignment_issues = (
    reference_duplicates
    + extraction_duplicates
)

PATHS["alignment_issues"].write_text(
    json.dumps(
        alignment_issues,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("Alignment issues:", len(alignment_issues))

if reference_duplicates:
    raise AssertionError(
        "Reference identity is not unique."
    )

Alignment issues: 0


In [14]:
# ============================================================
# 9. Match records and compare fields
# ============================================================

def compare_field(field, reference_value, extracted_value):
    if field == "Value":
        return (
            parse_numeric(reference_value)
            == parse_numeric(extracted_value)
        )

    if field == "Unit":
        return (
            canonical_null(reference_value)
            == canonical_null(extracted_value)
        )

    if field == "Reporting Period":
        return (
            canonical_period(reference_value)
            == canonical_period(extracted_value)
        )

    if field == "Source Location":
        return (
            canonical_source_location(reference_value)
            == canonical_source_location(extracted_value)
        )

    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )


detailed_rows = []
discrepant_rows = []
missing_rows = []
unsupported_rows = []

all_identities = sorted(
    set(reference_index.keys())
    | set(extraction_index.keys()),
    key=lambda x: (x is None, str(x))
)

for identity in all_identities:
    ref_entry = reference_index.get(identity)
    ext_entry = extraction_index.get(identity)

    if ref_entry is not None and ext_entry is None:
        row = ref_entry["record"].copy()
        row["Reference Record Index"] = ref_entry["record_index"]
        missing_rows.append(row)
        continue

    if ext_entry is not None and ref_entry is None:
        row = ext_entry["record"].copy()
        row["Extracted Record Index"] = ext_entry["record_index"]
        unsupported_rows.append(row)
        continue

    ref_record = ref_entry["record"]
    ext_record = ext_entry["record"]

    field_results = {
        field: compare_field(
            field,
            ref_record.get(field),
            ext_record.get(field)
        )
        for field in EXPECTED_FIELDS
    }

    primary_correct = all(
        field_results[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    # Identity agreement is guaranteed by the matched key, but retained
    # in the detailed output for transparency.
    identity_correct = all(
        field_results[field]
        for field in IDENTITY_FIELDS
    )

    fully_correct = (
        primary_correct
        and identity_correct
    )

    detail = {
        "Reference Record Index":
            ref_entry["record_index"],
        "Extracted Record Index":
            ext_entry["record_index"],
        "Identity Reporting Period":
            identity,
        "Fully Correct":
            fully_correct,
        "Primary Correct":
            primary_correct
    }

    for field in EXPECTED_FIELDS:
        detail[f"Reference {field}"] = ref_record.get(field)
        detail[f"Extracted {field}"] = ext_record.get(field)
        detail[f"{field} Correct"] = field_results[field]

    detailed_rows.append(detail)

    if not fully_correct:
        discrepant_rows.append(detail.copy())


detailed_df = pd.DataFrame(detailed_rows)
discrepant_df = pd.DataFrame(discrepant_rows)
missing_df = pd.DataFrame(missing_rows)
unsupported_df = pd.DataFrame(unsupported_rows)

print("Aligned:", len(detailed_df))
print("Discrepant:", len(discrepant_df))
print("Missing:", len(missing_df))
print("Unsupported/unmatched:", len(unsupported_df))

Aligned: 25
Discrepant: 0
Missing: 0
Unsupported/unmatched: 0


In [15]:
# ============================================================
# 10. Calculate common validation metrics
# ============================================================

reference_count = len(reference_df)
extracted_count = len(extracted_records)
aligned_count = len(detailed_df)

fully_correct_count = (
    int(detailed_df["Fully Correct"].sum())
    if aligned_count
    else 0
)

discrepant_count = len(discrepant_df)
missing_count = len(missing_df)
unsupported_count = len(unsupported_df)

completeness = (
    aligned_count / reference_count
    if reference_count
    else None
)

record_precision_exact = (
    fully_correct_count / extracted_count
    if extracted_count
    else 0.0
)

record_recall_exact = (
    fully_correct_count / reference_count
    if reference_count
    else 0.0
)

record_f1_exact = (
    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )
    if (
        record_precision_exact
        + record_recall_exact
    )
    else 0.0
)

primary_field_accuracy = {}

for field in PRIMARY_CORRECTNESS_FIELDS:
    if aligned_count:
        primary_field_accuracy[field] = float(
            detailed_df[
                f"{field} Correct"
            ].mean()
        )
    else:
        primary_field_accuracy[field] = None

valid_primary_values = [
    value
    for value in primary_field_accuracy.values()
    if value is not None
]

overall_primary_field_accuracy = (
    sum(valid_primary_values)
    / len(valid_primary_values)
    if valid_primary_values
    else None
)

category_metrics = {}

for category in sorted(
    set(reference_df["Category"].astype(str))
):
    ref_category_count = int(
        (reference_df["Category"] == category).sum()
    )

    ext_category_count = sum(
        1
        for record in extracted_records
        if isinstance(record, dict)
        and record.get("Category") == category
    )

    aligned_category = detailed_df[
        detailed_df["Reference Category"] == category
    ] if aligned_count else pd.DataFrame()

    aligned_category_count = len(aligned_category)

    fully_correct_category = (
        int(aligned_category["Fully Correct"].sum())
        if aligned_category_count
        else 0
    )

    p = (
        fully_correct_category / ext_category_count
        if ext_category_count
        else 0.0
    )
    r = (
        fully_correct_category / ref_category_count
        if ref_category_count
        else 0.0
    )
    f1 = (
        2 * p * r / (p + r)
        if p + r
        else 0.0
    )

    category_metrics[category] = {
        "expected_records": ref_category_count,
        "extracted_records": ext_category_count,
        "aligned_records": aligned_category_count,
        "fully_correct_records": fully_correct_category,
        "discrepant_records":
            aligned_category_count - fully_correct_category,
        "completeness":
            aligned_category_count / ref_category_count
            if ref_category_count
            else None,
        "record_precision_exact": p,
        "record_recall_exact": r,
        "record_f1_exact": f1
    }

In [16]:
# ============================================================
# 11. Create final Branch C validation summary
# ============================================================

comparison_rules = {
    "raw_extraction_modified":
        False,

    "manual_correction_applied":
        False,

    "comparison_normalisation_scope":
        "Comparison copies only",

    "identity_fields":
        IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "value":
        (
            "Exact represented numeric equality after deterministic "
            "parsing; no tolerance, rescaling, interpolation, rounding "
            "or conversion."
        ),

    "unit":
        (
            "Exact null agreement. No unit is inferred because the "
            "source CSV contains no explicit measurement-unit field."
        ),

    "text_fields":
        (
            "Conservative normalised exact agreement for fixed "
            "Category, Topic and Description values."
        ),

    "source_location":
        "Exact physical CSV data-row agreement.",

    "d12_equivalence_rules_status":
        (
            "Final D12 Validation A identity and comparison rules reused "
            "unchanged for Branch C. No Branch-C-specific semantic "
            "equivalence or performance-driven matching rule was added."
        ),

    "equivalence_rules_frozen":
        True
}


matching_rules = {
    "identity_fields":
        IDENTITY_FIELDS,

    "one_to_one_assignment":
        "Unique deterministic Reporting Period (year) identity.",

    "value_used_for_alignment":
        False,

    "unit_used_for_alignment":
        False,

    "category_used_for_alignment":
        False,

    "topic_used_for_alignment":
        False,

    "description_used_for_alignment":
        False,

    "source_location_used_for_alignment":
        False,

    "matching_rules_frozen_from_branch_A":
        True
}


content_diagnostics = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "reference_category_counts_valid":
        bool(reference_category_counts_valid),

    "extraction_record_count_valid":
        bool(
            extracted_count
            == EXPECTED_RECORD_COUNT
        ),

    "extraction_category_counts_valid":
        dict(
            Counter(
                record.get("Category")
                for record in extracted_records
                if isinstance(record, dict)
            )
        ) == EXPECTED_CATEGORY_COUNTS,

    "branch_C_scope_complete":
        structure_payload.get(
            "scope_complete"
        ),

    "branch_C_content_diagnostics":
        structure_payload.get(
            "content_diagnostics"
        ),

    "reference_identity_unique":
        bool(reference_identity_unique),

    "extraction_duplicate_identity_count":
        int(len(extraction_duplicates)),

    "ambiguous_identity_group_count":
        int(len(alignment_issues))
}


schema_diagnostics = {
    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_correct":
        bool(document_id_correct),

    "branch_correct":
        bool(branch_correct),

    "records_is_list":
        bool(records_is_list),

    "record_schema_valid":
        bool(record_schema_valid),

    "field_types_valid":
        bool(field_types_valid),

    "branch_C_structure_valid":
        bool(branch_C_structure_valid),

    "schema_validity":
        bool(schema_validity)
}


VALIDATION_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "reference_records":
        reference_count,

    "extracted_records":
        extracted_count,

    "aligned_records":
        aligned_count,

    "fully_correct_records":
        fully_correct_count,

    "discrepant_records":
        discrepant_count,

    "missing_records":
        missing_count,

    "unsupported_extracted_records":
        unsupported_count,

    "completeness":
        completeness,

    "missing_rate":
        (
            missing_count / reference_count
            if reference_count
            else None
        ),

    "record_precision_exact":
        record_precision_exact,

    "record_recall_exact":
        record_recall_exact,

    "record_f1_exact":
        record_f1_exact,

    "unsupported_rate":
        (
            unsupported_count / extracted_count
            if extracted_count
            else None
        ),

    "discrepancy_rate_among_aligned":
        (
            discrepant_count / aligned_count
            if aligned_count
            else None
        ),

    "overall_primary_field_accuracy":
        overall_primary_field_accuracy,

    "primary_field_accuracy":
        primary_field_accuracy,

    "schema_validity":
        bool(schema_validity),

    "schema_diagnostics":
        schema_diagnostics,

    "content_diagnostics":
        content_diagnostics,

    "branch_C_representation_integrity":
        representation_integrity,

    "matching_rules":
        matching_rules,

    "comparison_rules":
        comparison_rules,

    "reference_integrity_confirmation": {
        "reference_semantics_valid":
            bool(reference_semantics_valid),

        "checks":
            reference_semantic_checks,

        "reference_modified_by_validation":
            False
    },

    "category_metrics":
        category_metrics,

    "input_provenance":
        input_provenance,

    "comparison_rules_frozen_from_branch_A":
        True,

    "validation_timestamp":
        datetime.now(
            timezone.utc
        ).isoformat()
}


VALIDATION_CONCLUSION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "validation_status":
        (
            "Completed without discrepancies"
            if (
                fully_correct_count
                == reference_count
                and missing_count == 0
                and unsupported_count == 0
                and schema_validity
            )
            else
            "Completed with discrepancies"
        ),

    "reference_records":
        reference_count,

    "extracted_records":
        extracted_count,

    "aligned_records":
        aligned_count,

    "fully_correct_records":
        fully_correct_count,

    "discrepant_records":
        discrepant_count,

    "missing_records":
        missing_count,

    "unsupported_extracted_records":
        unsupported_count,

    "completeness":
        completeness,

    "record_precision_exact":
        record_precision_exact,

    "record_recall_exact":
        record_recall_exact,

    "record_f1_exact":
        record_f1_exact,

    "overall_primary_field_accuracy":
        overall_primary_field_accuracy,

    "schema_valid":
        bool(schema_validity),

    "normalisation_integrity_passed":
        bool(normalisation_integrity_passed),

    "comparison_rules_frozen_from_branch_A":
        True
}


print(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D12",
  "document_name": "Our World in Data — Annual CO2 emissions time series",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised structural Markdown table",
  "reference_records": 25,
  "extracted_records": 25,
  "aligned_records": 25,
  "fully_correct_records": 25,
  "discrepant_records": 0,
  "missing_records": 0,
  "unsupported_extracted_records": 0,
  "completeness": 1.0,
  "missing_rate": 0.0,
  "record_precision_exact": 1.0,
  "record_recall_exact": 1.0,
  "record_f1_exact": 1.0,
  "unsupported_rate": 0.0,
  "discrepancy_rate_among_aligned": 0.0,
  "overall_primary_field_accuracy": 1.0,
  "primary_field_accuracy": {
    "Category": 1.0,
    "Topic": 1.0,
    "Description": 1.0,
    "Value": 1.0,
    "Unit": 1.0,
    "Source Location": 1.0
  },
  "schema_validity": true,
  "schema_diagnostics": {
    "top_level_object_valid": true,
    "document_id_correct": true,
    "branch_corr

In [17]:
# ============================================================
# 12. Export validation outputs and final consistency checks
# ============================================================

detailed_df.to_csv(
    PATHS["detailed"],
    index=False,
    encoding="utf-8-sig"
)

discrepant_df.to_csv(
    PATHS["discrepant"],
    index=False,
    encoding="utf-8-sig"
)

missing_df.to_csv(
    PATHS["missing"],
    index=False,
    encoding="utf-8-sig"
)

unsupported_df.to_csv(
    PATHS["unsupported"],
    index=False,
    encoding="utf-8-sig"
)


PATHS["summary"].write_text(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


PATHS["conclusion"].write_text(
    json.dumps(
        VALIDATION_CONCLUSION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


missing_outputs = [
    path.name
    for path in PATHS.values()
    if not path.exists()
]

if missing_outputs:
    raise AssertionError(
        f"Missing validation outputs: {missing_outputs}"
    )


# ------------------------------------------------------------
# Methodological/provenance checks only.
# Do NOT assert perfect extraction performance.
# ------------------------------------------------------------

if not reference_semantics_valid:
    raise AssertionError(
        "Reference semantic validation failed."
    )

if not schema_validity:
    raise AssertionError(
        "Branch C extraction schema validation failed."
    )

if not branch_C_structure_valid:
    raise AssertionError(
        "Branch C structure check failed."
    )

if not parent_equivalence_passed:
    raise AssertionError(
        "Branch C parent-B equivalence failed."
    )

if not normalisation_integrity_passed:
    raise AssertionError(
        "Branch C normalisation integrity failed."
    )

if not metadata_source_hash_matches_stage_1:
    raise AssertionError(
        "Branch C metadata source hash does not match "
        "the frozen Stage 1 source identity."
    )

if not parsed_extraction_hash_matches_metadata:
    raise AssertionError(
        "Parsed extraction hash does not match "
        "Branch C experiment metadata."
    )


# ------------------------------------------------------------
# Validator-accounting checks.
# ------------------------------------------------------------

assert (
    aligned_count
    + missing_count
    == reference_count
)

assert (
    aligned_count
    + unsupported_count
    == extracted_count
)

assert (
    fully_correct_count
    + discrepant_count
    == aligned_count
)


print("Validation C — D12 completed successfully.")
print("Aligned records:", aligned_count)
print("Fully correct records:", fully_correct_count)
print("Discrepant records:", discrepant_count)
print("Missing records:", missing_count)
print(
    "Unsupported/unmatched records:",
    unsupported_count
)
print("Exact F1:", record_f1_exact)
print("Schema validity:", schema_validity)
print(
    "Normalisation integrity:",
    normalisation_integrity_passed
)

print("\nGenerated files:")

for path in PATHS.values():
    print("-", path.name)

for path in PATHS.values():
    files.download(path)

Validation C — D12 completed successfully.
Aligned records: 25
Fully correct records: 25
Discrepant records: 0
Missing records: 0
Unsupported/unmatched records: 0
Exact F1: 1.0
Schema validity: True
Normalisation integrity: True

Generated files:
- D12_branch_C_validation_detailed.csv
- D12_branch_C_discrepant_records.csv
- D12_branch_C_missing_records.csv
- D12_branch_C_unsupported_records.csv
- D12_branch_C_alignment_issues.json
- D12_reference_semantics_confirmation.json
- D12_branch_C_validation_summary.json
- D12_branch_C_validation_conclusion.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>